# CSCI 3151 — Foundations of Machine Learning
## M48 – Sequence modelling & vanilla RNNs

In this in-class mini-lab, you'll work with a synthetic **random walk** sequence task designed to stress-test the *memory* of recurrent networks.

Each example is a short 1‑D time series (a noisy random walk). The **target** is the *starting value* of the walk, which appears only at the very beginning of the sequence. We'll compare:

- a **feedforward MLP** that only sees a short, recent window of the sequence, and  
- a **vanilla RNN** that sees the *entire* sequence and can, in principle, remember important information from early time steps.

Although this is a toy DGP, it's a good stand‑in for real problems where **early events in a sequence matter later** (e.g., patient history, conversation context, long‑horizon forecasting).

**This mini-lab reinforces…**

- Distinguishing i.i.d. tabular data from **sequence data** where order matters.
- The **vanilla RNN** architecture and many‑to‑one sequence regression.
- How recurrence lets models access **long-range dependencies** that windowed MLPs struggle with.
- Reading and interpreting **training / validation curves** for sequence models.


In [ ]:
# Setup: imports and global settings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 3151

# Reproducibility
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# Core task 1: build a random-walk sequence dataset

SEQ_LEN   = 30       # number of time steps in each sequence
SIGMA     = 0.05     # per-step noise scale
N_TOTAL   = 6000     # total examples (train + val + test)
N_TRAIN   = 4000
N_VAL     = 1000
N_TEST    = N_TOTAL - N_TRAIN - N_VAL
WINDOW_MLP = 5       # how many trailing steps the MLP gets to see

rng = np.random.default_rng(SEED)

def generate_random_walks(n_samples, seq_len=SEQ_LEN, sigma=SIGMA, rng=None):
    """Generate (n_samples, seq_len) random-walk sequences and targets x0.

    Each sequence x_0, ..., x_{T-1} starts at a random x_0 ~ U(0,1)
    and evolves via x_t = x_{t-1} + eps_t, eps_t ~ N(0, sigma^2).
    Target y is the starting value x_0.
    """
    if rng is None:
        rng = np.random.default_rng()
    x0 = rng.uniform(0.0, 1.0, size=(n_samples,))
    walks = np.zeros((n_samples, seq_len), dtype=np.float32)
    walks[:, 0] = x0
    eps = rng.normal(loc=0.0, scale=sigma, size=(n_samples, seq_len - 1))
    for t in range(1, seq_len):
        walks[:, t] = walks[:, t - 1] + eps[:, t - 1]
    targets = x0.astype(np.float32).reshape(-1, 1)
    return walks, targets

walks, targets = generate_random_walks(N_TOTAL, SEQ_LEN, SIGMA, rng=rng)

# Train / val / test split
indices = rng.permutation(N_TOTAL)
train_idx = indices[:N_TRAIN]
val_idx   = indices[N_TRAIN:N_TRAIN + N_VAL]
test_idx  = indices[N_TRAIN + N_VAL:]

walks_train, y_train = walks[train_idx], targets[train_idx]
walks_val,   y_val   = walks[val_idx],   targets[val_idx]
walks_test,  y_test  = walks[test_idx],  targets[test_idx]

print("Walks shape (train / val / test):")
print(walks_train.shape, walks_val.shape, walks_test.shape)


In [ ]:
# Quick visual check: a few random walks

num_examples_to_plot = 6
fig, ax = plt.subplots(figsize=(8, 3))

for i in range(num_examples_to_plot):
    series = walks_train[i]
    ax.plot(range(SEQ_LEN), series, alpha=0.7)
    # highlight starting value x0 and the MLP's window
    ax.scatter(0, series[0], color="black", s=35, zorder=3)
    ax.axvspan(SEQ_LEN - WINDOW_MLP, SEQ_LEN - 1, alpha=0.1, color="orange")

ax.set_xlabel("Time step t")
ax.set_ylabel("x_t")
ax.set_title("Example random-walk sequences\n(starting value at t=0; shaded region = MLP window)")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


### Core task 1 — A windowed MLP baseline

The **MLP baseline** only sees the last few values of each sequence:

- Input features: the last `WINDOW_MLP` steps, e.g. `x_{T-5}, …, x_{T-1}`.
- Target: the starting value `x_0`.

After many noisy steps, the last few values carry very little information about `x_0`,
so we expect the MLP's best possible mean-squared error (MSE) to be much larger than the
per-step noise variance `σ²`.

Your goals in this section:

1. Understand how the dataset is turned into PyTorch tensors and dataloaders.
2. Train a small MLP and inspect its learning curves.
3. Compare the MLP's **test MSE** to the noise level `σ²` and to the RNN later.


In [ ]:
# Build PyTorch datasets for the MLP (windowed inputs)

X_mlp_train = walks_train[:, -WINDOW_MLP:]   # (N_train, WINDOW_MLP)
X_mlp_val   = walks_val[:,   -WINDOW_MLP:]
X_mlp_test  = walks_test[:,  -WINDOW_MLP:]

X_mlp_train_t = torch.from_numpy(X_mlp_train).float()
X_mlp_val_t   = torch.from_numpy(X_mlp_val).float()
X_mlp_test_t  = torch.from_numpy(X_mlp_test).float()

y_train_t = torch.from_numpy(y_train).float()
y_val_t   = torch.from_numpy(y_val).float()
y_test_t  = torch.from_numpy(y_test).float()

batch_size = 64

mlp_train_ds = TensorDataset(X_mlp_train_t, y_train_t)
mlp_val_ds   = TensorDataset(X_mlp_val_t,   y_val_t)
mlp_test_ds  = TensorDataset(X_mlp_test_t,  y_test_t)

mlp_train_loader = DataLoader(mlp_train_ds, batch_size=batch_size, shuffle=True)
mlp_val_loader   = DataLoader(mlp_val_ds,   batch_size=batch_size)
mlp_test_loader  = DataLoader(mlp_test_ds,  batch_size=batch_size)

X_mlp_train_t.shape, y_train_t.shape


In [ ]:
# Define a simple MLP regressor on the windowed inputs

class WindowMLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

mlp_model = WindowMLPRegressor(input_dim=WINDOW_MLP, hidden_dim=32).to(device)
mlp_model


In [ ]:
# Generic training loop helper (we'll reuse this for the RNN)

def train_model(model, train_loader, val_loader, epochs=40, lr=1e-3, device=device):
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"epoch": [], "train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        # training
        model.train()
        running_train_loss = 0.0
        n_train = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            preds = model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * xb.size(0)
            n_train += xb.size(0)

        avg_train_loss = running_train_loss / n_train

        # validation
        model.eval()
        running_val_loss = 0.0
        n_val = 0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                preds = model(xb)
                loss = loss_fn(preds, yb)
                running_val_loss += loss.item() * xb.size(0)
                n_val += xb.size(0)

        avg_val_loss = running_val_loss / n_val

        history["epoch"].append(epoch)
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d} | train MSE={avg_train_loss:.5f} | val MSE={avg_val_loss:.5f}")

    return pd.DataFrame(history)


In [ ]:
# Train the MLP baseline

mlp_history = train_model(mlp_model, mlp_train_loader, mlp_val_loader, epochs=60, lr=1e-3)

mlp_history.tail()


In [ ]:
# Plot MLP learning curves

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(mlp_history["epoch"], mlp_history["train_loss"], label="train MSE")
ax.plot(mlp_history["epoch"], mlp_history["val_loss"],   label="val MSE")
ax.axhline(SIGMA**2, color="green", linestyle="--", label=r"noise level $\\sigma^2$")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.set_title("MLP on last window only")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# Evaluate MLP on train / val / test sets

def evaluate_regressor(model, data_loader, device=device):
    model.eval()
    loss_fn = nn.MSELoss()
    running_loss = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            running_loss += loss.item() * xb.size(0)
            n += xb.size(0)
    return running_loss / n

mlp_train_mse = evaluate_regressor(mlp_model, mlp_train_loader)
mlp_val_mse   = evaluate_regressor(mlp_model, mlp_val_loader)
mlp_test_mse  = evaluate_regressor(mlp_model, mlp_test_loader)

print(f"MLP train MSE: {mlp_train_mse:.5f}")
print(f"MLP val   MSE: {mlp_val_mse:.5f}")
print(f"MLP test  MSE: {mlp_test_mse:.5f}")
print(f"Noise level sigma^2 ≈ {SIGMA**2:.5f}")


### Core task 2 — A vanilla RNN that sees the whole sequence

Now we give a model access to the **entire random walk**:

- Inputs: the full sequence `x_0, x_1, …, x_{T-1}` (as a sequence of scalars).
- Model: a vanilla RNN with `tanh` nonlinearity.
- Output: a single regression prediction for `x_0`, based on the RNN's **final hidden state**.

Your goals in this section:

1. Build the sequence dataset for the RNN (shape `(N, T, 1)`).
2. Train a small RNN and compare its learning curves and MSE to the MLP baseline.
3. Reflect on *why* the RNN can do better on this task.


In [ ]:
# Build PyTorch datasets for the RNN (full sequence inputs)

# RNN expects shape (batch, seq_len, input_size)
X_rnn_train = walks_train[:, :, np.newaxis]   # add a feature dimension of size 1
X_rnn_val   = walks_val[:,   :, np.newaxis]
X_rnn_test  = walks_test[:,  :, np.newaxis]

X_rnn_train_t = torch.from_numpy(X_rnn_train).float()
X_rnn_val_t   = torch.from_numpy(X_rnn_val).float()
X_rnn_test_t  = torch.from_numpy(X_rnn_test).float()

rnn_train_ds = TensorDataset(X_rnn_train_t, y_train_t)
rnn_val_ds   = TensorDataset(X_rnn_val_t,   y_val_t)
rnn_test_ds  = TensorDataset(X_rnn_test_t,  y_test_t)

rnn_train_loader = DataLoader(rnn_train_ds, batch_size=batch_size, shuffle=True)
rnn_val_loader   = DataLoader(rnn_val_ds,   batch_size=batch_size)
rnn_test_loader  = DataLoader(rnn_test_ds,  batch_size=batch_size)

X_rnn_train_t.shape, y_train_t.shape


In [ ]:
# Define a simple vanilla RNN regressor

class SimpleRNNRegressor(nn.Module):
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            nonlinearity="tanh",
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        outputs, h_last = self.rnn(x)
        # h_last: (num_layers * num_directions, batch, hidden_size)
        last_hidden = h_last[-1]          # (batch, hidden_size)
        out = self.fc(last_hidden)        # (batch, 1)
        return out

rnn_model = SimpleRNNRegressor(input_size=1, hidden_size=32).to(device)
rnn_model


In [ ]:
# Train the RNN on full sequences

rnn_history = train_model(rnn_model, rnn_train_loader, rnn_val_loader, epochs=60, lr=1e-3)

rnn_history.tail()


In [ ]:
# Compare learning curves: MLP vs RNN

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(mlp_history["epoch"], mlp_history["val_loss"], label="MLP val MSE (window only)")
ax.plot(rnn_history["epoch"], rnn_history["val_loss"], label="RNN val MSE (full sequence)")
ax.axhline(SIGMA**2, color="green", linestyle="--", label=r"noise level $\\sigma^2$")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.set_title("Validation performance: windowed MLP vs vanilla RNN")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# Evaluate RNN on train / val / test sets

rnn_train_mse = evaluate_regressor(rnn_model, rnn_train_loader)
rnn_val_mse   = evaluate_regressor(rnn_model, rnn_val_loader)
rnn_test_mse  = evaluate_regressor(rnn_model, rnn_test_loader)

summary = pd.DataFrame({
    "model": ["MLP (window)", "RNN (full seq)"],
    "train_mse": [mlp_train_mse, rnn_train_mse],
    "val_mse":   [mlp_val_mse,   rnn_val_mse],
    "test_mse":  [mlp_test_mse,  rnn_test_mse],
})

summary


### Core task 3 — Peeking inside the RNN's hidden state

Vanilla RNNs compress the whole history of the sequence into their **hidden state**.
In this toy task, the network has to *store something about `x_0`* in its hidden state
and preserve it through many noisy steps.

In this section you'll:

1. Inspect the RNN's hidden state across time for a few sequences.
2. See how the **final hidden state** relates to the starting value `x_0`.
3. Connect what you see back to the unrolled computation graph from the video.

Run the code cell and then try changing which examples you inspect.


In [ ]:
# Inspect hidden states for a few test sequences

rnn_model.eval()

num_to_inspect = 4
example_indices = np.random.choice(len(walks_test), size=num_to_inspect, replace=False)

fig, axes = plt.subplots(1, num_to_inspect, figsize=(4 * num_to_inspect, 3), sharey=True)

with torch.no_grad():
    for ax, idx in zip(axes, example_indices):
        seq_np = walks_test[idx:idx+1, :, np.newaxis]   # shape (1, T, 1)
        x0_true = y_test[idx, 0]

        seq = torch.from_numpy(seq_np).float().to(device)
        outputs, h_last = rnn_model.rnn(seq)           # we bypass rnn_model.forward() to get all states
        # outputs: (1, T, hidden_size)
        hidden_traj = outputs[0].cpu().numpy()         # (T, hidden_size)

        # For visualization, just plot the first hidden dimension over time
        ax.plot(range(SEQ_LEN), hidden_traj[:, 0], marker="o", alpha=0.8)
        ax.set_xlabel("t")
        ax.set_title(f"Example idx {idx}\\ntrue x0 ≈ {x0_true:.2f}")
        ax.grid(True, alpha=0.2)

axes[0].set_ylabel("h_t[0] (first hidden dim)")
plt.suptitle("RNN hidden state trajectory (first dimension) on a few test sequences")
plt.tight_layout()
plt.show()


## Stretch ideas (if you have time)

These are **optional** and aimed at faster / more curious students.  
You do *not* need to complete them during class, but they make good follow‑ups.

1. **Vary the MLP's window size.**  
   - Try `WINDOW_MLP = 3, 5, 10, 20` (regenerate inputs, re-init the MLP, and retrain).  
   - How does the test MSE change? Does the MLP ever match the RNN?

2. **Make the task harder or easier.**  
   - Increase or decrease `SEQ_LEN` or the noise level `SIGMA`.  
   - When sequences get longer or noisier, which model degrades faster?

3. **Probe generalisation to different sequence lengths.**  
   - Train on `SEQ_LEN = 30` but evaluate both models on sequences with `SEQ_LEN = 10` or `SEQ_LEN = 60`.  
   - Does the RNN cope better with **out-of-distribution horizons** than the MLP?

4. **Relate back to BPTT and gradient issues.**  
   - As you make sequences longer, watch the RNN's learning curves.  
   - When (if at all) do you start to see signs of vanishing / exploding gradients in this vanilla RNN?

You can start by copying the existing training code into new cells and modifying the hyperparameters.


## Wrap-up

In this mini-lab you:

- Built a **synthetic sequence dataset** where the important signal (`x_0`) appears only at the **start** of each sequence.
- Trained a **windowed MLP** and saw that, even with a good optimizer, it cannot beat the information-theoretic limits of its short input window.
- Trained a **vanilla RNN** on full sequences and observed that it can achieve much lower MSE by storing information about `x_0` in its hidden state.
- Visualised the RNN's **hidden trajectory** over time, connecting it back to the unrolled computation graph from the video.

This mirrors the broader lesson from the module:

> When **long-range dependencies** matter, architectures with **recurrence** (or other sequence-aware structure) are essential — feedforward networks over fixed windows are often fundamentally limited.
